# Object detection 2: YOLO

**YOLO** (*You Only Look Once*) looks at the whole image once: a grid over the image predicts, for every cell at the same time, whether an object is centred there, its box and its class; overlapping duplicates are then removed. It is fast enough for video on a CPU.

This notebook fine-tunes an Ultralytics YOLO model (`YOLO_MODEL`, pretrained on COCO's 80 everyday classes) on `data_object/`: Ultralytics handles the augmentation (mosaic, colour, scale), the training and picking the best epoch on the val split. The test split is scored once at the end, by Ultralytics and by `imgcls.objdet.metrics` (the code that scores notebook 03), so the two detectors compare on equal terms.

Licence: Ultralytics is AGPL-3.0 (fine for learning and course work; closed-source products need their commercial licence).

**On Kaggle:** attach the `data_object` dataset made by notebook 01 with *Add Input*, and turn on a GPU and *Internet* in the side panel (the setup cell downloads the project code from GitHub, so it runs the **pushed** version of `imgcls/`). Runs are saved in `/kaggle/working/output`.

Generated by `scripts/create_detection_notebooks.py` on 2026-11-24. To reset this notebook: `python scripts/create_detection_notebooks.py --only 02 --force`

In [ ]:
# ---- settings for this model (edit freely) ----
MODEL_NAME = 'yolo26n'          # results go to output/<MODEL_NAME>/<timestamp>/
YOLO_MODEL = 'yolo26n.pt'       # pretrained on COCO; sizes n/s/m/l/x (yolo26s.pt is ~3x slower, more accurate)
EPOCHS = 50
IMG_SIZE = 640                  # training resolution; 320 trains about 4x faster on a CPU
BATCH_SIZE = 16                 # -1 = as many as fit in 60% of the GPU memory; 8 for a 4 GB GPU
FRACTION = 1.0                  # share of the training images used, e.g. 0.2 for a quick CPU try
PATIENCE = 15                   # stop after this many epochs without val improvement
NUM_WORKERS = 2                 # image-loading processes; 0 if loading hangs on Windows
DEVICE = 'auto'                 # "auto" = GPU if available, otherwise CPU
CONF = 0.25                     # minimum score for the boxes drawn on pictures and videos (not used for mAP)
VIDEO_PATH = None               # a video to run the detector on, e.g. r"C:\Users\me\Downloads\clip.mp4"
VIDEO_FPS = 2.0                 # frames analysed per second of video
OBJECT_DIR = None               # None = find data_object/ (project folder; Kaggle: attached dataset)
SEED = 42

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

# find the project (the folder holding imgcls/); on Colab / Kaggle fetch it from GitHub first
REPO_URL = "https://github.com/M0HiT27/Image-Classification.git"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imgcls").is_dir()), None)
if ROOT is None:
    on_colab = Path("/content").is_dir() and not Path("/kaggle").is_dir()  # Kaggle has /content too
    ROOT = Path("/content" if on_colab else "/tmp") / "Image-Classification"
    if (ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))
# forget an imgcls loaded earlier in this session, so re-running this cell after a pull uses the new code
for name in [m for m in sys.modules if m == "imgcls" or m.startswith("imgcls.")]:
    del sys.modules[name]


def ensure(module, pip_name=None):
    """Install a package that is missing (Kaggle and Colab have most of them already)."""
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)


ensure('timm')
ensure('ultralytics')
ensure('cv2', 'opencv-python-headless')

from IPython.display import Image, display
import imgcls
from imgcls import objdet

from imgcls.objdet import dataset, viz, yolo

env, object_dir = objdet.setup(OBJECT_DIR)
imgcls.set_seed(SEED)
device = imgcls.get_device(DEVICE)

## Train
Ultralytics prints one block per epoch and saves `weights/best.pt` (best val mAP) and `weights/last.pt`. Interrupting the cell keeps the best weights so far.

In [ ]:
cfg = imgcls.config_from_globals(globals())
run_dir = yolo.train(cfg, object_dir, device)
if (run_dir / "results.png").exists():
    display(Image(filename=str(run_dir / "results.png")))

## Evaluate on the test split
**mAP50**: a box counts as found when it overlaps the true box by at least 50% (IoU >= 0.5); **mAP50-95** averages IoU thresholds 0.50 to 0.95, so it also rewards tight boxes. 1.0 is perfect. Most test boxes come from OWLv2, so this measures agreement with OWLv2 for those classes (human boxes for banana, orange and mushroom).

In [ ]:
# to evaluate an earlier run instead: run_dir = imgcls.OUTPUT_DIR / MODEL_NAME / "<timestamp>"
table = yolo.evaluate_run(run_dir, object_dir, device, "test", IMG_SIZE, BATCH_SIZE if BATCH_SIZE > 0 else 16)
display(table)
if (run_dir / "test" / "confusion_matrix.png").exists():
    display(Image(filename=str(run_dir / "test" / "confusion_matrix.png")))

## Look at predictions
Random test images: coloured boxes are predictions (score >= `CONF`), thin white ones the labels.

In [ ]:
predict = yolo.predictor(run_dir, device, IMG_SIZE, CONF)
grid = viz.predictions_grid(dataset.read_split(object_dir, "test"), predict, run_dir / "test_examples.png")
display(Image(filename=str(grid)))

## Run it on a video
Writes `video_annotated.mp4` and a sheet of frames into the run folder.

In [ ]:
if VIDEO_PATH:
    video, sheet, counts = viz.annotate_video(VIDEO_PATH, predict, run_dir / "video_annotated.mp4", VIDEO_FPS)
    display(Image(filename=str(sheet)))

## Compare detection runs
Every YOLO and Faster R-CNN run in the runs folder, newest first.

In [ ]:
display(objdet.detection_runs())